# QuantBench 模型矩阵 Smoke 回测

目标：验证 `src/model` 的 QuantBench 模型 registry 不只是清单，而是可以通过统一 adapter 完成训练、预测、回测和 `src/analysis` 对比。

说明：本 notebook 是 smoke 层验证。当前 adapter 用于固定全平台接口和 sidecar 产物格式；逐个模型的论文级严格实现可以后续在不改 contract 的前提下替换。

In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.model.matrix import run_quantbench_matrix_smoke

CARD_DIR = REPO_ROOT / "research" / "quantbench_model_matrix_smoke_v1"
outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
)
outputs.summary

{'status': 'smoke_completed',
 'model_count': 32,
 'start': '2026-01-01',
 'end': '2026-06-09',
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08',
 'top_model': 'logistic_clf',
 'adapter_status': 'smoke_adapter',
 'note': 'Smoke adapters validate the full matrix contract; paper-strict replacements can be swapped per ModelSpec.'}

In [2]:
outputs.model_comparison.head(10)

,rank,model,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,1,logistic_clf,49.032788,62.147247,-13.114459,21.677243,3.853465,0.740764,3.128315,54.214258
1,2,random_forest_clf,49.032788,62.147247,-13.114459,21.677243,3.853465,0.740764,3.128315,54.214258
2,3,alstm_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
3,4,autoformer_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
4,5,darnn_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
5,6,dlinear_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
6,7,fedformer_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
7,8,gat_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
8,9,gcn_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407
9,10,gru_mse,44.648453,62.147247,-17.498794,18.743499,4.202316,0.598022,6.458415,50.325407


In [3]:
outputs.regime_metrics.head(12)

,model,regime,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,alstm_mse,bull,23.079005,82.228080,-59.149075,2.676199,9.559105,0.686786,1.863233,111.598337
1,alstm_mse,downtrend,45.561573,65.770282,-20.208709,6.817155,11.914085,0.725562,2.515358,151.150263
2,alstm_mse,neutral,0.000000,0.000000,0.000000,0.000000,0.000000,0.287022,0.000000,0.000000
3,alstm_mse,stress,2.397114,32.768915,-30.371802,5.756710,2.102953,0.358805,2.079824,12.071109
4,autoformer_mse,bull,23.079005,82.228080,-59.149075,2.676199,9.559105,0.686786,1.863233,111.598337
5,autoformer_mse,downtrend,45.561573,65.770282,-20.208709,6.817155,11.914085,0.725562,2.515358,151.150263
6,autoformer_mse,neutral,0.000000,0.000000,0.000000,0.000000,0.000000,0.287022,0.000000,0.000000
7,autoformer_mse,stress,2.397114,32.768915,-30.371802,5.756710,2.102953,0.358805,2.079824,12.071109
8,catboost_rank,bull,60.111124,94.603219,-34.492095,2.804272,12.008870,0.664091,1.672625,173.349238
9,catboost_rank,downtrend,64.054299,105.958443,-41.904144,0.000000,13.638200,0.709408,0.868382,198.699533


## 中文结论

- 做了什么：执行 QuantBench 模型矩阵 smoke 训练、预测、预测策略回测和 `src/analysis` 横向分析。
- 核心指标：以本目录 `analysis_summary.json`、`model_comparison.csv`、`window_metrics.csv`、`regime_metrics.csv` 为准。
- regime 差异：已输出 `regime_metrics.csv`，按 bull、neutral、downtrend、stress 等状态比较收益、回撤、换手和稳定性。
- 是否晋升：否。本轮只证明全模型矩阵 contract 与 sidecar 记录路径可运行，不代表任何模型可进入 production。
- 风险：当前 `adapter_status=smoke_adapter`，不是逐篇论文严格复现；后续要逐模型替换 adapter 并保留同一 manifest/schema。
- 下一步：补充模型 artifact manifest、论文级 adapter 优先级、以及 dashboard 对 `model_comparison.csv` 的只读展示。